# Strip GNN-skipped UniProt IDs from 3DCNN test JSONL logs

**Source**: `catGNN_test_272997.out` lines 262–321 (`[SKIP] <UniProt>`).

**Input**: all `*.jsonl` under `results/test/` (skips filenames ending with `.delete.jsonl`).

**Output**: `{stem}.delete.jsonl` beside each source.

**Logic**: Rows store parallel arrays `sample_ids`, `probs`, `preds`, `labels` with ids such as `A0Q7Q2.pdb|A|123`. Indices whose UniProt is in SKIP are removed from **all four** arrays; scalar fields (`batch_size`, TP/TN/FP/FN, rates, BCE loss, optional AUROC/AUPRC) are recomputed for the filtered minibatch.

**Note**: Per-batch metrics only reflect retained samples inside that batch, not dataset-level aggregates.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
from typing import Dict, Iterable, Optional, Tuple

import numpy as np

# catGNN_test_272997.out lines 262–321 ([SKIP] accessions)
SKIP_ACCESSIONS = frozenset(
    [
        "A0Q7Q2",
        "C9X1G5",
        "H9T8G6",
        "J3F2B0",
        "J7RUA5",
        "P0CQ11",
        "Q01804",
        "Q03LF7",
        "Q07794",
        "Q6NKI3",
        "Q6ZPJ3",
        "Q927P4",
        "Q9D8C3",
        "Q9ZVX1",
        "U2UMQ6",
    ]
)

TEST_RESULTS_DIR = Path(
    "/data/data3/conglab/s441865/code/Catalytic_sites_setA_3dcnn/results/test"
)

In [ ]:
def normalize_json_nan_line(line: str) -> str:
    """Upstream logs emit bare NaN tokens; coerce to JSON null for stdlib json.loads."""
    line = re.sub(r":\s*-?Infinity\b", ": null", line)
    return re.sub(r":\s*NaN\b", ": null", line)


def accession_from_sample_id(sid: str) -> Optional[str]:
    """ACCESSION.pdb|CHAIN|resnum"""
    head = sid.split("|", 1)[0]
    if not head.endswith(".pdb"):
        return None
    return head[:-4]


def recompute_au_metrics(labels: np.ndarray, probs: np.ndarray) -> Tuple[float, float]:
    """Mean AP & ROC-AUC per batch only if sklearn is present and labels are mixed."""
    if labels.size == 0 or len(np.unique(labels)) < 2:
        return float("nan"), float("nan")
    try:
        from sklearn.metrics import average_precision_score, roc_auc_score
    except ImportError:
        return float("nan"), float("nan")
    y = labels.astype(np.float64).ravel()
    p = probs.astype(np.float64).ravel()
    return float(average_precision_score(y, p)), float(roc_auc_score(y, p))


def recompute_classification_inplace(row: Dict) -> None:
    probs = np.asarray(row["probs"], dtype=np.float64)
    preds = np.asarray(row["preds"], dtype=np.float64)
    labs = np.asarray(row["labels"], dtype=np.float64)
    n = probs.size

    if n != labs.size or n != preds.size:
        raise ValueError("Lengths disagree after filtering")

    row["batch_size"] = int(n)

    if n == 0:
        nan = float("nan")
        row.update(
            {
                "loss": nan,
                "acc": nan,
                "precision": nan,
                "recall": nan,
                "f1": nan,
                "sensitivity": nan,
                "specificity": nan,
                "auprc": nan,
                "auroc": nan,
                "tp": 0,
                "tn": 0,
                "fp": 0,
                "fn": 0,
            }
        )
        return

    y = labs >= 0.5
    yhat = preds >= 0.5

    tp = int(np.sum(yhat & y))
    tn = int(np.sum((~yhat) & (~y)))
    fp = int(np.sum(yhat & (~y)))
    fn = int(np.sum((~yhat) & y))

    eps = 1e-12
    bce = -(
        labs * np.log(np.clip(probs, eps, 1.0 - eps))
        + (1.0 - labs) * np.log(np.clip(1.0 - probs, eps, 1.0 - eps))
    )
    row["loss"] = float(np.mean(bce))

    acc = float((tp + tn) / n)
    prec = float(tp / max(tp + fp, 1)) if (tp + fp) > 0 else 0.0
    rec = float(tp / max(tp + fn, 1)) if (tp + fn) > 0 else 0.0
    if prec + rec == 0.0:
        f1 = 0.0
    else:
        f1 = float(2 * prec * rec / (prec + rec))

    spec = float(tn / max(tn + fp, 1)) if (tn + fp) > 0 else 0.0
    auprc, auroc = recompute_au_metrics(labs, probs)

    row.update(
        {
            "acc": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "sensitivity": rec,
            "specificity": spec,
            "auprc": auprc,
            "auroc": auroc,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
        }
    )


def filter_parallel_lists(row: Dict) -> Dict:
    out = dict(row)
    ids = list(row.get("sample_ids") or ())
    probs = list(row.get("probs") or ())
    preds = list(row.get("preds") or ())
    labels = list(row.get("labels") or ())
    lengths = {len(ids), len(probs), len(preds), len(labels)}
    if len(lengths) != 1:
        raise ValueError("Inconsistent list lengths: %s" % lengths)

    keep = []
    for i, sid in enumerate(ids):
        acc = accession_from_sample_id(sid)
        if acc is None or acc not in SKIP_ACCESSIONS:
            keep.append(i)

    idx = np.array(keep, dtype=int)
    if idx.size:
        out["sample_ids"] = [ids[int(i)] for i in idx]
        out["probs"] = [float(probs[int(i)]) for i in idx]
        out["preds"] = [float(preds[int(i)]) for i in idx]
        out["labels"] = [float(labels[int(i)]) for i in idx]
    else:
        out["sample_ids"] = []
        out["probs"] = []
        out["preds"] = []
        out["labels"] = []

    recompute_classification_inplace(out)
    return out


def dumps_row_stable(row: Dict) -> str:
    return json.dumps(row, separators=(",", ": "), allow_nan=True)


def rel_under_test_results(p: Path) -> str:
    try:
        return str(p.relative_to(TEST_RESULTS_DIR))
    except ValueError:
        return str(p)

In [ ]:
def iter_input_jsonls(root: Path) -> Iterable[Path]:
    for p in sorted(root.rglob("*.jsonl")):
        if p.name.endswith(".delete.jsonl"):
            continue
        if p.stem.endswith(".delete"):
            continue
        yield p


def process_jsonl_file(src_path: Path) -> Path:
    dst = src_path.with_name("{}.delete.jsonl".format(src_path.stem))

    n_in_lines = n_out_lines = bad_json = 0
    removed_positions = 0
    batches_cleared = batches_partial = 0

    with src_path.open("r", encoding="utf-8") as fin, dst.open(
        "w", encoding="utf-8"
    ) as fout:
        for line in fin:
            stripped = line.strip()
            if not stripped:
                continue
            n_in_lines += 1
            try:
                row = json.loads(normalize_json_nan_line(stripped))
            except json.JSONDecodeError as exc:
                bad_json += 1
                print("bad JSON [%s #%s]: %s" % (src_path.name, n_in_lines, exc))
                continue

            n_before = len(row.get("sample_ids") or ())
            filt = filter_parallel_lists(row)
            n_after = len(filt["sample_ids"])
            removed_positions += max(0, n_before - n_after)
            if n_before:
                if n_after == 0:
                    batches_cleared += 1
                elif n_after < n_before:
                    batches_partial += 1

            fout.write(dumps_row_stable(filt) + "\n")
            n_out_lines += 1

    print(
        "%s -> %s   lines_written=%s   removed_positions=%s   "
        "batches_emptied=%s   batches_partially_removed=%s   "
        "json_decode_errors=%s"
        % (
            rel_under_test_results(src_path),
            dst.name,
            n_out_lines,
            removed_positions,
            batches_cleared,
            batches_partial,
            bad_json,
        )
    )
    return dst


def run_all():
    paths = list(iter_input_jsonls(TEST_RESULTS_DIR))
    print("Filtering %s jsonl file(s)\n" % len(paths))
    return [process_jsonl_file(p) for p in paths]


created = run_all()
created